In [4]:
"""
Stack Overflow 2025 Survey - Data Cleaning & Preprocessing
===========================================================
This notebook prepares the raw survey data for all downstream analyses.
Team members should run this ONCE before starting individual work.

Author: Data Foundation Lead
Date: 2025
"""

import pandas as pd
import numpy as np
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

RAW_DATA_PATH = 'data/raw/survey_results_public.csv'
SCHEMA_PATH = 'data/raw/survey_results_schema.csv'
CLEANED_OUTPUT = 'data/processed/cleaned_full.csv'
FEATURED_OUTPUT = 'data/processed/feature_engineered.csv'

# Attention check - users must select 'Apples'
ATTENTION_CHECK_VALUE = 'Apples'

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("=" * 80)
print("STACK OVERFLOW 2025 SURVEY - DATA PREPROCESSING")
print("=" * 80)

print("\n[1/8] Loading raw data...")
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"   ✓ Loaded {len(df):,} responses with {len(df.columns)} columns")

# Load schema for reference
schema = pd.read_csv(SCHEMA_PATH)
print(f"   ✓ Loaded schema with {len(schema)} question definitions")

initial_count = len(df)

# ============================================================================
# STEP 2: ATTENTION CHECK FILTERING
# ============================================================================

print("\n[2/8] Filtering attention check failures...")
if 'Check' in df.columns:
    valid_mask = df['Check'] == ATTENTION_CHECK_VALUE
    failed_count = (~valid_mask).sum()
    df = df[valid_mask].copy()
    print(f"   ✓ Removed {failed_count} respondents who failed attention check")
    print(f"   ✓ Remaining: {len(df):,} responses ({len(df)/initial_count*100:.1f}%)")
else:
    print("   ⚠ Warning: 'Check' column not found, skipping attention filter")

# ============================================================================
# STEP 3: HANDLE MISSING VALUES
# ============================================================================

print("\n[3/8] Analyzing and handling missing values...")

# Critical columns that need data for most analyses
critical_cols = ['CompTotal', 'YearsCodePro', 'DevType', 'Country']
missing_critical = df[critical_cols].isnull().sum()

print("\n   Missing values in critical columns:")
for col in critical_cols:
    if col in df.columns:
        missing_pct = (df[col].isnull().sum() / len(df)) * 100
        print(f"     - {col}: {df[col].isnull().sum():,} ({missing_pct:.1f}%)")

# For CompTotal: Keep missing as NaN (users can filter in their analyses)
# For categorical: Create 'Unknown' category
for col in ['DevType', 'RemoteWork', 'EdLevel']:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna('Unknown')

# ============================================================================
# STEP 4: DATA TYPE CONVERSIONS
# ============================================================================

print("\n[4/8] Converting data types...")

# Convert CompTotal to numeric (remove any non-numeric characters)
if 'CompTotal' in df.columns:
    df['CompTotal'] = pd.to_numeric(df['CompTotal'], errors='coerce')
    print(f"   ✓ Converted CompTotal to numeric")

# Convert experience columns
for col in ['YearsCode', 'YearsCodePro', 'WorkExp']:
    if col in df.columns:
        # Handle special values like "Less than 1 year" or "More than 50 years"
        df[col] = df[col].astype(str)
        df[col] = df[col].replace({
            'Less than 1 year': '0.5',
            'More than 50 years': '50'
        })
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"   ✓ Converted {col} to numeric")

# ============================================================================
# STEP 5: HANDLE MULTI-SELECT COLUMNS
# ============================================================================

print("\n[5/8] Processing multi-select columns...")

# Identify multi-select columns (contain semicolons in responses)
multi_select_patterns = [
    'HaveWorkedWith', 'WantToWorkWith', 'Admired',  # Tech stack
    'Employment', 'CodingActivities', 'LearnCode', 'LearnCodeOnline',
    'BuyNewTool', 'AIBen', 'AIEthics', 'AIChallenges', 'Frustration',
    'ProfessionalTech', 'SOHow'
]

multi_select_cols = [col for col in df.columns 
                     if any(pattern in col for pattern in multi_select_patterns)]

print(f"   ✓ Identified {len(multi_select_cols)} multi-select columns")

# Function to one-hot encode multi-select
def expand_multiselect(df, column):
    """Convert semicolon-separated values into binary columns"""
    if column not in df.columns:
        return df
    
    # Get unique values across all responses
    all_values = set()
    for val in df[column].dropna():
        if isinstance(val, str) and ';' in val:
            all_values.update([v.strip() for v in val.split(';')])
    
    # Create binary columns for each unique value
    for value in all_values:
        new_col_name = f"{column}_{value.replace(' ', '_').replace('/', '_')}"
        df[new_col_name] = df[column].apply(
            lambda x: 1 if isinstance(x, str) and value in x else 0
        )
    
    return df

# Apply to key technology columns for easier analysis
tech_cols = [col for col in multi_select_cols if 'Language' in col or 'Database' in col 
             or 'Platform' in col or 'Webframe' in col]

print(f"   ⓘ One-hot encoding {len(tech_cols)} technology columns...")
print("     (This may take a moment...)")

for col in tech_cols[:5]:  # Start with top 5 most important
    if col in df.columns:
        df = expand_multiselect(df, col)
        
print(f"   ✓ Expanded multi-select columns (added ~{len([c for c in df.columns if '_' in c])} binary features)")

# ============================================================================
# STEP 6: FEATURE ENGINEERING
# ============================================================================

print("\n[6/8] Engineering derived features...")

# 1. Salary Quartiles
if 'CompTotal' in df.columns:
    df['salary_quartile'] = pd.qcut(
        df['CompTotal'], 
        q=4, 
        labels=['Q1', 'Q2', 'Q3', 'Q4'],
        duplicates='drop'
    )
    print("   ✓ Created salary_quartile (Q1-Q4)")

# 2. High Earner (75th percentile)
if 'CompTotal' in df.columns:
    threshold = df['CompTotal'].quantile(0.75)
    df['high_earner'] = (df['CompTotal'] >= threshold).astype(int)
    print(f"   ✓ Created high_earner (>= ${threshold:,.0f})")

# 3. Experience Categories
if 'YearsCodePro' in df.columns:
    df['experience_category'] = pd.cut(
        df['YearsCodePro'],
        bins=[-1, 2, 5, 10, 100],
        labels=['Junior (0-2)', 'Mid (3-5)', 'Senior (6-10)', 'Expert (10+)']
    )
    print("   ✓ Created experience_category (Junior/Mid/Senior/Expert)")

# 4. Remote Work Binary
if 'RemoteWork' in df.columns:
    df['is_remote'] = df['RemoteWork'].apply(
        lambda x: 1 if isinstance(x, str) and 'remote' in x.lower() else 0
    )
    print("   ✓ Created is_remote (binary)")

# 5. AI User Binary
if 'AISelect' in df.columns:
    df['is_ai_user'] = (df['AISelect'] == 'Yes').astype(int)
    print("   ✓ Created is_ai_user (binary)")

# 6. AI Sentiment Score (for easier analysis)
if 'AISent' in df.columns:
    sentiment_map = {
        'Very favorable': 2,
        'Favorable': 1,
        'Indifferent': 0,
        'Unsure': 0,
        'Very unfavorable': -2
    }
    df['ai_sentiment_score'] = df['AISent'].map(sentiment_map)
    print("   ✓ Created ai_sentiment_score (-2 to +2)")

# 7. Job Satisfaction Score
if 'JobSat' in df.columns:
    df['job_satisfaction_score'] = pd.to_numeric(df['JobSat'], errors='coerce')
    print("   ✓ Created job_satisfaction_score (numeric)")

# 8. Stack Overflow Engagement Level
so_cols = ['SOVisitFreq', 'SOPartFreq', 'SOComm']
if all(col in df.columns for col in so_cols):
    # Create engagement score based on frequency
    freq_score = {
        'Multiple times per day': 5,
        'Daily or almost daily': 4,
        'A few times per week': 3,
        'A few times per month or weekly': 2,
        'Less than once per month or monthly': 1,
        'I have never participated in Q&A on Stack Overflow': 0
    }
    
    df['so_engagement_score'] = df['SOPartFreq'].map(freq_score).fillna(0)
    print("   ✓ Created so_engagement_score (0-5)")

# 9. Age Group
if 'Age' in df.columns:
    age_order = [
        'Under 18 years old',
        '18-24 years old', 
        '25-34 years old',
        '35-44 years old',
        '45-54 years old',
        '55-64 years old',
        '65 years or older',
        'Prefer not to say'
    ]
    df['age_group'] = pd.Categorical(df['Age'], categories=age_order, ordered=True)
    print("   ✓ Created age_group (ordered categorical)")

# 10. Education Level (simplified)
if 'EdLevel' in df.columns:
    edu_map = {
        'Primary/elementary school': 'Basic',
        'Secondary school': 'Basic',
        'Some college/university': 'Undergraduate',
        'Associate degree': 'Undergraduate',
        "Bachelor's degree": 'Undergraduate',
        "Master's degree": 'Graduate',
        'Professional degree': 'Graduate',
        'Something else': 'Other'
    }
    df['education_level'] = df['EdLevel'].apply(
        lambda x: edu_map.get(x, 'Unknown') if pd.notna(x) else 'Unknown'
    )
    print("   ✓ Created education_level (simplified)")

# ============================================================================
# STEP 7: DATA QUALITY CHECKS & OUTLIER HANDLING
# ============================================================================

print("\n[7/8] Performing data quality checks...")

# Winsorize extreme salaries (cap at 1st and 99th percentile)
if 'CompTotal' in df.columns:
    p1 = df['CompTotal'].quantile(0.01)
    p99 = df['CompTotal'].quantile(0.99)
    
    outliers_low = (df['CompTotal'] < p1).sum()
    outliers_high = (df['CompTotal'] > p99).sum()
    
    df['CompTotal_winsorized'] = df['CompTotal'].clip(lower=p1, upper=p99)
    
    print(f"   ✓ Winsorized CompTotal (capped {outliers_low} low, {outliers_high} high outliers)")
    print(f"     Range: ${p1:,.0f} - ${p99:,.0f}")

# Check for duplicate ResponseIds
if 'ResponseId' in df.columns:
    duplicates = df['ResponseId'].duplicated().sum()
    if duplicates > 0:
        print(f"   ⚠ Warning: Found {duplicates} duplicate ResponseIds")
        df = df.drop_duplicates(subset=['ResponseId'], keep='first')
    else:
        print("   ✓ No duplicate ResponseIds found")

# ============================================================================
# STEP 8: SAVE PROCESSED DATA
# ============================================================================

print("\n[8/8] Saving processed datasets...")

# Create output directory if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

# Save cleaned data
df.to_csv(CLEANED_OUTPUT, index=False)
print(f"   ✓ Saved cleaned data to: {CLEANED_OUTPUT}")
print(f"     Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Save feature-engineered version (subset with key features)
feature_cols = [
    'ResponseId', 'MainBranch', 'Age', 'age_group', 'Employment', 'RemoteWork', 'is_remote',
    'EdLevel', 'education_level', 'YearsCode', 'YearsCodePro', 'experience_category',
    'DevType', 'OrgSize', 'Country', 'Currency', 'CompTotal', 'CompTotal_winsorized',
    'salary_quartile', 'high_earner', 'AISelect', 'is_ai_user', 'AISent', 
    'ai_sentiment_score', 'AIAcc', 'AIThreat', 'JobSat', 'job_satisfaction_score',
    'SOVisitFreq', 'SOPartFreq', 'so_engagement_score', 'WorkExp', 'ICorPM'
]

# Only keep columns that exist
feature_cols_exist = [col for col in feature_cols if col in df.columns]
df_featured = df[feature_cols_exist].copy()

df_featured.to_csv(FEATURED_OUTPUT, index=False)
print(f"   ✓ Saved feature-engineered data to: {FEATURED_OUTPUT}")
print(f"     Shape: {df_featured.shape[0]:,} rows × {df_featured.shape[1]} columns")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("\n" + "=" * 80)
print("DATA PREPROCESSING COMPLETE - SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset Overview:")
print(f"   • Total valid responses: {len(df):,}")
print(f"   • Total features: {len(df.columns)}")
print(f"   • Response rate after filtering: {len(df)/initial_count*100:.1f}%")

print(f"\n💰 Compensation Statistics:")
if 'CompTotal' in df.columns:
    comp_stats = df['CompTotal'].describe()
    print(f"   • Responses with salary data: {df['CompTotal'].notna().sum():,} ({df['CompTotal'].notna().sum()/len(df)*100:.1f}%)")
    print(f"   • Mean: ${comp_stats['mean']:,.0f}")
    print(f"   • Median: ${comp_stats['50%']:,.0f}")
    print(f"   • Range: ${comp_stats['min']:,.0f} - ${comp_stats['max']:,.0f}")

print(f"\n🎯 Key Demographics:")
if 'DevType' in df.columns:
    print(f"   • Top 3 Developer Types:")
    for role, count in df['DevType'].value_counts().head(3).items():
        print(f"     - {role}: {count:,} ({count/len(df)*100:.1f}%)")

if 'experience_category' in df.columns:
    print(f"   • Experience Distribution:")
    for exp, count in df['experience_category'].value_counts().sort_index().items():
        print(f"     - {exp}: {count:,} ({count/len(df)*100:.1f}%)")

print(f"\n🤖 AI Adoption:")
if 'is_ai_user' in df.columns:
    ai_users = df['is_ai_user'].sum()
    print(f"   • Using AI tools: {ai_users:,} ({ai_users/len(df)*100:.1f}%)")
    print(f"   • Not using AI tools: {len(df) - ai_users:,} ({(len(df) - ai_users)/len(df)*100:.1f}%)")

print(f"\n🌍 Geographic Distribution:")
if 'Country' in df.columns:
    print(f"   • Top 5 Countries:")
    for country, count in df['Country'].value_counts().head(5).items():
        print(f"     - {country}: {count:,} ({count/len(df)*100:.1f}%)")

print("\n" + "=" * 80)
print("✅ PREPROCESSING COMPLETE - Ready for individual analyses!")
print("=" * 80)

print("\n📝 Next Steps:")
print("   1. Review data/processed/cleaned_full.csv")
print("   2. Check data/processed/feature_engineered.csv for quick analysis")
print("   3. Proceed with individual notebook analyses")
print("   4. Document any additional cleaning steps in methodology.md")

STACK OVERFLOW 2025 SURVEY - DATA PREPROCESSING

[1/8] Loading raw data...
   ✓ Loaded 65,437 responses with 114 columns
   ✓ Loaded schema with 87 question definitions

[2/8] Filtering attention check failures...
   ✓ Removed 0 respondents who failed attention check
   ✓ Remaining: 65,437 responses (100.0%)

[3/8] Analyzing and handling missing values...

   Missing values in critical columns:
     - CompTotal: 31,697 (48.4%)
     - YearsCodePro: 13,827 (21.1%)
     - DevType: 5,992 (9.2%)
     - Country: 6,507 (9.9%)

[4/8] Converting data types...
   ✓ Converted CompTotal to numeric
   ✓ Converted YearsCode to numeric
   ✓ Converted YearsCodePro to numeric
   ✓ Converted WorkExp to numeric

[5/8] Processing multi-select columns...
   ✓ Identified 44 multi-select columns
   ⓘ One-hot encoding 12 technology columns...
     (This may take a moment...)
   ✓ Expanded multi-select columns (added ~238 binary features)

[6/8] Engineering derived features...
   ✓ Created salary_quartile (Q1-